# Bab 1 — Eksplorasi dan Analisis Data

**Buku:** *Practical Statistics for Data Scientists* (O'Reilly)

## Ringkasan Bab
Analisis Data Eksploratif (EDA) merupakan langkah awal yang esensial dalam setiap proyek sains data. Sebelum membangun model prediktif, kita perlu mengenali karakteristik dasar data: nilai sentral, tingkat penyebaran, bentuk distribusi, serta keterkaitan antar variabel. Bab ini membahas empat pilar utama:

1. **Ukuran Pemusatan** - merangkum nilai pusat data (mean, trimmed mean, median, dan versi berbobot).
2. **Ukuran Variabilitas** - mengukur seberapa rapat atau renggang sebaran data (varians, standar deviasi, MAD, IQR, dan persentil).
3. **Distribusi Data** - visualisasi sebaran melalui boxplot, tabel frekuensi, histogram, density plot, dan diagram batang.
4. **Korelasi dan Hubungan** - analisis hubungan antar variabel menggunakan korelasi, scatterplot, hexagonal binning, heatmap, dan contour plot.

Seluruh kode di bawah ini adalah implementasi mandiri dari materi bab ini menggunakan dataset pendamping buku.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import trim_mean, median_abs_deviation
import wquantiles
import seaborn as sns
import matplotlib.pyplot as plt

DATA = Path('..') / 'data'
sns.set_theme(style='whitegrid')

## 1. Ukuran Pemusatan (Estimates of Location)

**Teori.** Ukuran pemusatan bertujuan merangkum nilai sentral dari suatu variabel agar mudah diinterpretasikan.

| Ukuran | Definisi Singkat | Sensitif terhadap Pencilan? |
|---|---|---|
| Mean | Jumlah total nilai dibagi n | Tidak |
| Trimmed mean | Mean setelah membuang persentase tertentu dari nilai ekstrem | Ya (sebagian besar teredam) |
| Weighted mean | Penjumlahan berbobot sum(w_i * x_i) / sum(w_i) | Tidak |
| Median | Nilai tengah dari data yang telah diurutkan | Ya |
| Weighted median | Nilai di mana akumulasi bobot di bawah dan atas bernilai seimbang | Ya |

Mean memperhitungkan seluruh observasi sehingga efisien, namun rentan terhadap nilai ekstrem. Median bersifat *robust* karena hanya bergantung pada bagian tengah data. Trimmed mean menjadi kompromi yang baik dengan membuang ekstrem di kedua ujung. Bobot digunakan manakala observasi memiliki tingkat kepentingan yang berbeda.

**Dataset:** `state.csv` - data populasi dan tingkat pembunuhan di berbagai negara bagian AS.

In [ ]:
state = pd.read_csv(DATA / 'state.csv')
state.head(8)

In [ ]:
print('Mean:            ', state['Population'].mean())
print('Trimmed mean 10%:', trim_mean(state['Population'], 0.1))
print('Median:          ', state['Population'].median())

Mean populasi (sekitar 6,16 juta) lebih tinggi dari median (sekitar 4,44 juta) karena tertarik ke atas oleh beberapa negara bagian berpenduduk sangat besar. Hal ini mengindikasikan bahwa distribusi data menceng ke kanan (*right-skewed*).

Untuk **tingkat pembunuhan** (*murder rate*), karena ukuran populasi antar negara bagian bervariasi, penggunaan mean berbobot berdasarkan populasi memberikan representasi yang lebih akurat.

In [ ]:
print('Mean murder rate:          ', state['Murder.Rate'].mean())
print('Weighted mean murder rate: ', np.average(state['Murder.Rate'], weights=state['Population']))
print('Weighted median:           ', wquantiles.median(state['Murder.Rate'], weights=state['Population']))

## 2. Ukuran Variabilitas (Estimates of Variability)

**Teori.** Ukuran pemusatan menunjukkan pusat data, sementara variabilitas (dispersi) mengukur seberapa rapat atau tersebarnya nilai-nilai di sekitar pusat tersebut.

* **Varians:** Rata-rata kuadrat deviasi dari mean (dibagi n-1 untuk estimasi tak bias).
* **Standar Deviasi:** Akar kuadrat dari varians (memiliki satuan yang setara dengan data).
* **Mean Absolute Deviation (MAD):** Rata-rata deviasi mutlak dari mean.
* **Median Absolute Deviation (MAD):** Median dari deviasi mutlak terhadap median (sangat *robust*).
* **Range, Persentil, dan IQR:** Rentang antarkuartil (IQR = persentil 75 - persentil 25) yang tahan terhadap pencilan.

In [ ]:
pop = state['Population']
print('Std dev:            ', pop.std())
print('IQR:                ', pop.quantile(0.75) - pop.quantile(0.25))
print('MAD (scaled, normal):', median_abs_deviation(pop, scale='normal'))

Standar deviasi (~6,85 juta) jauh lebih besar dibandingkan ukuran berbasis IQR dan MAD, kembali menunjukkan dampak dari negara bagian berpenduduk masif.

### Persentil dan Boxplot
**Persentil** menunjukkan ambang batas nilai di mana persentase tertentu data berada di bawahnya. **Boxplot** memvisualisasikan kuartil, rentang *whiskers* hingga 1,5 x IQR, serta mengidentifikasi pencilan di luar batas tersebut.

In [ ]:
print(state['Murder.Rate'].quantile([0.05, 0.25, 0.5, 0.75, 0.95]))

In [ ]:
ax = (state['Population'] / 1_000_000).plot.box(figsize=(3, 4))
ax.set_ylabel('Population (millions)')
plt.tight_layout(); plt.show()

### Tabel Frekuensi dan Histogram
**Tabel frekuensi** membagi rentang variabel menjadi interval (*bin*) seragam dan menghitung jumlah observasi pada tiap interval. **Histogram** adalah representasi visual dari tabel frekuensi tersebut.

In [ ]:
binned = pd.cut(state['Population'], 10)
print(binned.value_counts().sort_index())

In [ ]:
ax = (state['Population'] / 1_000_000).plot.hist(figsize=(5, 4), bins=10, edgecolor='white')
ax.set_xlabel('Population (millions)')
plt.tight_layout(); plt.show()

### Estimasi Densitas (Density Estimates)
**Density plot** menyajikan bentuk penghalusan dari histogram menggunakan estimasi densitas kernel (*Kernel Density Estimation* / KDE) untuk mengeliminasi kearbitraran batas interval.

In [ ]:
ax = state['Murder.Rate'].plot.hist(density=True, xlim=[0, 12], bins=range(1, 12), edgecolor='white', figsize=(5, 4))
state['Murder.Rate'].plot.density(ax=ax)
ax.set_xlabel('Murder rate (per 100,000)')
plt.tight_layout(); plt.show()

## 3. Eksplorasi Data Biner dan Kategorikal

**Teori.** Untuk variabel kategorikal, ukuran utama yang digunakan adalah proporsi dan modus. **Diagram batang** (*bar chart*) menjadi visualisasi standar untuk menunjukkan frekuensi tiap kategori secara diskret.

**Dataset:** `dfw_airline.csv` - rekapitulasi penyebab keterlambatan penerbangan di bandara Dallas/Fort Worth.

In [ ]:
dfw = pd.read_csv(DATA / 'dfw_airline.csv')
print(dfw.T)

In [ ]:
ax = dfw.transpose().plot.bar(figsize=(5, 4), legend=False)
ax.set_xlabel('Cause of delay')
ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

Keterlambatan pesawat sebelumnya (`Inbound`) merupakan kontributor terbesar dan modus utama, disusul oleh kontrol lalu lintas udara (`ATC`) dan maskapai penerbangan.

## 4. Korelasi

**Teori.** *Korelasi* mengukur tingkat keterkaitan pergerakan linier antara dua variabel numerik melalui koefisien Pearson (berkisar dari -1 hingga +1). Perlu dicatat bahwa korelasi bersifat sensitif terhadap pencilan, hanya mengukur hubungan linier, dan tidak menyiratkan hubungan kausalitas (sebab-akibat).

In [ ]:
sp500_sym = pd.read_csv(DATA / 'sp500_sectors.csv')
sp500_px = pd.read_csv(DATA / 'sp500_data.csv.gz', index_col=0)

telecom_symbols = sp500_sym[sp500_sym['sector'] == 'telecommunications_services']['symbol']
telecom = sp500_px.loc[sp500_px.index >= '2012-07-01', telecom_symbols]
telecom.corr()

In [ ]:
etfs = sp500_px.loc[sp500_px.index > '2012-07-01',
                    sp500_sym[sp500_sym['sector'] == 'etf']['symbol']]
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(etfs.corr(), vmin=-1, vmax=1,
            cmap=sns.diverging_palette(20, 220, as_cmap=True), ax=ax)
plt.tight_layout(); plt.show()

Sebagian besar ETF menunjukkan korelasi positif karena bergerak seirama dengan tren pasar secara umum, sementara ETF volatilitas (`VXX`) berkorelasi negatif.

### Scatterplot

In [ ]:
ax = telecom.plot.scatter(x='T', y='VZ', figsize=(4, 4), marker='$\u25EF$')
ax.set_xlabel('ATT (T)')
ax.set_ylabel('Verizon (VZ)')
ax.axhline(0, color='grey', lw=1)
ax.axvline(0, color='grey', lw=1)
plt.tight_layout(); plt.show()

## 5. Eksplorasi Dua Variabel atau Lebih

**Teori.** Ketika menghadapi dataset berukuran besar dengan titik data melimpah, scatterplot biasa dapat mengalami *overplotting*. Teknik alternatif meliputi **hexagonal binning** (pengelompokan sel heksagonal berbasis frekuensi) dan **contour plot** (garis kontur densitas). Untuk data bervariasi kategori dan numerik, digunakan boxplot serta violin plot.

**Dataset:** `kc_tax.csv.gz` - data taksiran pajak properti di King County, WA.

In [ ]:
kc_tax = pd.read_csv(DATA / 'kc_tax.csv.gz')
kc_tax0 = kc_tax.loc[(kc_tax.TaxAssessedValue < 750000) &
                     (kc_tax.SqFtTotLiving > 100) &
                     (kc_tax.SqFtTotLiving < 3500), :]
print(kc_tax0.shape)

In [ ]:
ax = kc_tax0.plot.hexbin(x='SqFtTotLiving', y='TaxAssessedValue',
                         gridsize=30, sharex=False, figsize=(5, 4))
ax.set_xlabel('Finished Square Feet')
ax.set_ylabel('Tax Assessed Value')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
sns.kdeplot(data=kc_tax0.sample(10000, random_state=1),
            x='SqFtTotLiving', y='TaxAssessedValue', ax=ax)
ax.set_xlabel('Finished Square Feet')
ax.set_ylabel('Tax Assessed Value')
plt.tight_layout(); plt.show()

### Dua Variabel Kategorikal
**Dataset:** `lc_loans.csv` - pinjaman Lending Club berdasarkan tingkat risiko (*grade*) dan status pelunasan.

In [ ]:
lc_loans = pd.read_csv(DATA / 'lc_loans.csv')
crosstab = lc_loans.pivot_table(index='grade', columns='status',
                                aggfunc=lambda x: len(x), margins=True)
print(crosstab)

In [ ]:
df = crosstab.copy().loc['A':'G', :].astype(float)
df.loc[:, 'Charged Off':'Late'] = df.loc[:, 'Charged Off':'Late'].div(df['All'], axis=0)
df['All'] = df['All'] / sum(df['All'])
perc_crosstab = df
print(perc_crosstab)

Proporsi gagal bayar (*charged-off*) dan keterlambatan meningkat secara konsisten dari grade A ke G, membuktikan bahwa grade merupakan indikator risiko yang valid.

### Data Kategorikal dan Numerik
**Dataset:** `airline_stats.csv` - proporsi keterlambatan penerbangan per maskapai.